# Building a RAG Chatbot for Ala-Too International University

## What is Retrieval-Augmented Generation (RAG)?

Imagine you're a student taking an open-book exam about Ala-Too University. You don't need to have memorized every fact — instead, a helper highlights the most relevant pages from the textbook and hands them to you. You read those pages and write your answer. That's **RAG**.

More precisely, RAG is a technique that gives a language model (LLM) access to an external knowledge base at question-answering time. Instead of relying on what the model learned during training (which may be outdated or incomplete), RAG:

1. **Retrieves** the most relevant passages from a curated knowledge base (in our case, the university website)
2. **Augments** the LLM's prompt with those passages ("here, read these before answering")
3. **Generates** an answer that is grounded in real, verifiable source material

The key benefit: the LLM cannot make up facts it hasn't been shown. If the answer isn't in the knowledge base, the model says so — rather than confidently hallucinating a wrong answer.

**Our pipeline:**
```
Website → scraper.py → /data/raw/*.txt
                             ↓
                        indexer.py → /data/index/
                             ↓
User question → chatbot.py → retrieve chunks → Mistral (HF API) → answer
```

This notebook walks through every step so you can see exactly what happens at each stage. **Run each cell in order.**

---
## Cell 1 — Install Dependencies and Imports

Here's what each library does:

| Library | Role |
|---|---|
| `llama-index-core` | The RAG framework: connects documents, embeddings, and LLMs |
| `llama-index-embeddings-huggingface` | Runs embedding models locally (no API key needed) |
| `llama-index-llms-huggingface` | Calls HuggingFace Inference API for the LLM |
| `sentence-transformers` | Runs `all-MiniLM-L6-v2` embedding model on your CPU |
| `requests` + `beautifulsoup4` | Download and parse web pages |
| `python-dotenv` | Loads `.env` config file into `os.environ` |
| `scikit-learn` | Cosine similarity math for the embedding demo |
| `numpy` | Fast array operations for vector math |

**First run:** installing takes 2-5 minutes and downloads ~500MB of packages.
**Subsequent runs:** packages are cached, this cell completes in seconds.

In [1]:
# Install all required packages.
# The -q flag suppresses verbose output so we only see errors.
# Remove -q if you want to see exactly what's being installed.
import sys
!{sys.executable} -m pip install -q \
    llama-index-core \
    llama-index-embeddings-huggingface \
    llama-index-llms-huggingface \
    llama-index-readers-web \
    sentence-transformers \
    huggingface_hub \
    transformers \
    requests beautifulsoup4 \
    python-dotenv \
    scikit-learn numpy

print("All packages installed!")

# ── Core Python standard library ────────────────────────────────────────────
import os
import re
import time
import urllib.parse
from collections import deque
from pathlib import Path

# ── Third-party: web scraping ───────────────────────────────────────────────
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv

# ── LlamaIndex: RAG framework ────────────────────────────────────────────────
from llama_index.core import (
    Settings,
    SimpleDirectoryReader,
    StorageContext,
    VectorStoreIndex,
    load_index_from_storage,
)
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface import HuggingFaceInferenceAPI

# ── Scientific computing (for the embedding deep dive in Cell 4) ─────────────
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# ── Load .env configuration ──────────────────────────────────────────────────
# Make sure you have a .env file with HF_TOKEN=hf_your_token_here
# Copy .env.example to .env and fill in your token.
load_dotenv()

print("All imports successful!")
print(f"HF_TOKEN set: {'Yes' if os.getenv('HF_TOKEN') else 'NO — please set it in .env!'}")

ERROR: Cannot install sentence-transformers==0.1.0, sentence-transformers==0.2.0, sentence-transformers==0.2.1, sentence-transformers==0.2.2, sentence-transformers==0.2.3, sentence-transformers==0.2.4, sentence-transformers==0.2.4.1, sentence-transformers==0.2.5, sentence-transformers==0.2.5.1, sentence-transformers==0.2.6.1, sentence-transformers==0.2.6.2, sentence-transformers==0.3.0, sentence-transformers==0.3.1, sentence-transformers==0.3.2, sentence-transformers==0.3.3, sentence-transformers==0.3.4, sentence-transformers==0.3.5, sentence-transformers==0.3.5.1, sentence-transformers==0.3.6, sentence-transformers==0.3.7, sentence-transformers==0.3.7.1, sentence-transformers==0.3.7.2, sentence-transformers==0.3.8, sentence-transformers==0.3.9, sentence-transformers==0.4.0, sentence-transformers==0.4.1, sentence-transformers==0.4.1.1, sentence-transformers==0.4.1.2, sentence-transformers==1.0.0, sentence-transformers==1.0.1, sentence-transformers==1.0.2, sentence-transformers==1.0.3, 

ModuleNotFoundError: No module named 'llama_index'

---
## Cell 2 — Scrape the Website (Build the Knowledge Base)

Before we can answer questions about Ala-Too University, we need to collect the source material. This cell crawls the university website and saves each page as a plain `.txt` file.

**What we're building:** a folder `data/raw/` with one `.txt` file per web page. Each file contains the clean text extracted from that page — navigation menus, scripts, and repeated footer content are stripped out.

**Why not just use the website live?** We could send the user's question to the LLM along with a URL and ask it to browse the web — but that would be slower, costlier, and less controlled. Pre-collecting pages lets us:
- Clean and normalize the text once
- Control which pages are in the knowledge base
- Embed the text once (not on every question)

Watch the output below — you'll see each page being downloaded in real time.

In [ ]:
# Import our scraper module (which we wrote in scraper.py)
# All the crawling logic lives there — this cell just calls it.
from scraper import crawl_website

# Run the crawler. Watch the output as pages are downloaded.
# This takes ~30-60 seconds for 20 pages (0.5s delay between requests).
pages_saved = crawl_website()

# Show what was collected
raw_data_dir = Path("data") / "raw"
saved_files = sorted(raw_data_dir.glob("*.txt"))

print(f"\n{'='*50}")
print(f"Pages saved: {pages_saved}")
print(f"\nFiles in data/raw/:")
total_chars = 0
for txt_file in saved_files:
    file_size = len(txt_file.read_text(encoding="utf-8"))
    total_chars += file_size
    print(f"  {txt_file.name[:60]:<60} {file_size:>8,} chars")

print(f"\nTotal text collected: {total_chars:,} characters")
print(f"Average per page:     {total_chars // max(len(saved_files), 1):,} characters")

# Peek at the first file to see what we saved
if saved_files:
    first_file_text = saved_files[0].read_text(encoding="utf-8")
    print(f"\n{'='*50}")
    print(f"Preview of: {saved_files[0].name}")
    print(f"{'='*50}")
    print(first_file_text[:500])
    print("... (truncated)")

---
## Cell 3 — Chunking: Splitting Text for the LLM

We now have raw text files — but we can't feed an entire page to the LLM for every question. Here's why, and what we do instead:

**The context window problem:**
Every LLM has a "context window" — the maximum amount of text it can process at once. Mistral-7B supports up to ~32,000 tokens (≈24,000 words). A typical Ala-Too webpage might have 2,000–5,000 characters, and we have 20 pages, giving us 40,000–100,000 characters total. That's too much for a single prompt, and even if it fit, sending all of it every time would be slow and expensive.

**The solution — chunking:**
We split each page into small overlapping segments called "chunks." For each user question, we find the 3 most relevant chunks and only send those to the LLM.

**Why overlap matters:**
Run the cell and look at the side-by-side chunks. Notice how the last ~50 characters of Chunk N appear at the start of Chunk N+1. This prevents information from being "lost" at a boundary.

```
Chunk N   : [━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━░░░]
Chunk N+1 :                                          [░░░━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━]
                                                      ↑
                                              overlap (50 chars)
```

In [ ]:
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter

# ── Configuration ────────────────────────────────────────────────────────────
CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", "512"))      # target chunk size in characters
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", "50")) # characters shared between adjacent chunks

# ── Load a sample document for inspection ───────────────────────────────────
# We load the first saved .txt file to demonstrate chunking.
raw_data_dir = Path("data") / "raw"
sample_files = sorted(raw_data_dir.glob("*.txt"))

if not sample_files:
    print("No files found in data/raw/ — please run Cell 2 first!")
else:
    # Read the file and wrap it in a LlamaIndex Document object
    sample_text = sample_files[0].read_text(encoding="utf-8")
    sample_document = Document(text=sample_text)

    # ── Create the splitter ───────────────────────────────────────────────────
    # SentenceSplitter tries to cut at sentence boundaries (. ! ?) rather than
    # mid-word. This keeps chunks more readable and coherent.
    sentence_splitter = SentenceSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )

    # ── Split the document into chunks (nodes) ────────────────────────────────
    nodes = sentence_splitter.get_nodes_from_documents([sample_document])

    print(f"Source file:    {sample_files[0].name}")
    print(f"Total chars:    {len(sample_text):,}")
    print(f"Chunk size:     {CHUNK_SIZE} chars")
    print(f"Chunk overlap:  {CHUNK_OVERLAP} chars")
    print(f"Total chunks:   {len(nodes)}")
    print(f"Average chunk:  {sum(len(n.get_content()) for n in nodes) // len(nodes):,} chars")

    # ── Show two consecutive chunks to visualize the overlap ─────────────────
    # Pick a chunk from the middle (not the first, which may be short)
    mid_idx = max(0, len(nodes) // 2 - 1)

    print(f"\n{'='*65}")
    print(f"CHUNK {mid_idx} (showing last 120 chars):")
    print(f"{'='*65}")
    chunk_a_text = nodes[mid_idx].get_content()
    print(f"...{chunk_a_text[-120:]}")

    print(f"\n{'='*65}")
    print(f"CHUNK {mid_idx + 1} (showing first 120 chars):")
    print(f"{'='*65}")
    chunk_b_text = nodes[mid_idx + 1].get_content()
    print(f"{chunk_b_text[:120]}...")

    # ── Find the overlap between the two chunks ───────────────────────────────
    # The last CHUNK_OVERLAP characters of chunk N should appear at the start
    # of chunk N+1 (approximately — SentenceSplitter adjusts to sentence boundaries).
    overlap_candidate = chunk_a_text[-CHUNK_OVERLAP:].strip()
    if overlap_candidate[:20] in chunk_b_text:
        print(f"\n✓ Overlap confirmed: chunk {mid_idx} tail appears in chunk {mid_idx+1} head")
        print(f"  Overlapping text: '{overlap_candidate[:40]}...'")
    else:
        print(f"\n(Note: SentenceSplitter adjusted boundaries to avoid mid-sentence cuts)")

    # ── Count chunks across ALL documents (for the full knowledge base) ────────
    print(f"\n{'='*65}")
    print("Chunking ALL saved pages (to estimate total index size):")
    all_documents = SimpleDirectoryReader(
        input_dir=str(raw_data_dir), required_exts=[".txt"]
    ).load_data()
    all_nodes = sentence_splitter.get_nodes_from_documents(all_documents)
    print(f"  Total documents : {len(all_documents)}")
    print(f"  Total chunks    : {len(all_nodes)}")
    print(f"  Each chunk will become one 384-dimensional embedding vector.")
    print(f"  Total vectors to store: {len(all_nodes)} × 384 = {len(all_nodes) * 384:,} floats")

---
## Cell 4 — Embeddings Deep Dive: Text → Numbers → Meaning

**What is an embedding?**

An embedding converts a piece of text into a list of numbers — a vector. But not random numbers: the numbers encode *meaning*. Two pieces of text with similar meaning get similar vectors; unrelated text gets very different vectors.

For example:
- `"university tuition fee"` → `[0.14, -0.23, 0.07, ...]` (384 numbers)
- `"how much does it cost to study"` → `[0.16, -0.21, 0.09, ...]` (384 numbers — very close!)
- `"the library opens at 9am"` → `[-0.31, 0.88, -0.42, ...]` (384 numbers — very different)

The model `all-MiniLM-L6-v2` was trained on millions of sentence pairs labeled "similar" or "different," learning to put similar sentences close together in this 384-dimensional space.

**How we measure "closeness":**
We use **cosine similarity**: a score from 0 (completely unrelated) to 1 (identical meaning). The formula measures the angle between two vectors.

**Expected results** (before running):
- Sentences 0 and 1 (both about fees): similarity **≈ 0.80–0.95** (very high)
- Sentences 0 and 2 (fees vs. library): similarity **≈ 0.20–0.50** (low)
- Sentences 1 and 2 (fees vs. library): similarity **≈ 0.20–0.50** (low)

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# ── Load the embedding model ──────────────────────────────────────────────────
# First run: downloads ~22MB to ~/.cache/huggingface/. Takes ~30s.
# Subsequent runs: loads from cache in ~2s.
print("Loading embedding model... (first run downloads ~22MB)")
embed_model = HuggingFaceEmbedding(model_name="all-MiniLM-L6-v2")
print("Model loaded!\n")

# ── Choose three sentences to compare ────────────────────────────────────────
# Sentences 0 and 1 are about the same topic (fees).
# Sentence 2 is about a completely different topic (library).
# This contrast demonstrates that embeddings capture MEANING, not just keywords.
test_sentences = [
    "What is the tuition fee at Ala-Too University?",        # S0: about fees (question form)
    "How much does it cost to study at Ala-Too University?", # S1: same topic, different words
    "The university library is open from 9am to 9pm daily.", # S2: unrelated topic
]

# ── Generate 384-dimensional vectors for each sentence ───────────────────────
print("Embedding 3 sentences...")
embedding_vectors = [
    embed_model.get_text_embedding(sentence)
    for sentence in test_sentences
]

# ── Show the first 8 dimensions of each vector ───────────────────────────────
print(f"\n{'='*65}")
print("First 8 of 384 dimensions for each sentence:")
print(f"{'='*65}")

for i, (sentence, vector) in enumerate(zip(test_sentences, embedding_vectors)):
    # Round to 4 decimal places for readable display
    first_8 = [round(v, 4) for v in vector[:8]]
    print(f"\nS{i}: '{sentence[:55]}...'")
    print(f"     Vector (dims 0-7): {first_8}")

# ── Compute cosine similarity between all pairs ───────────────────────────────
# cosine_similarity expects a 2D array: (n_sentences, n_dimensions)
vectors_matrix = np.array(embedding_vectors)  # shape: (3, 384)
similarity_matrix = cosine_similarity(vectors_matrix)  # shape: (3, 3)

print(f"\n{'='*65}")
print("Cosine Similarity Matrix (1.0 = identical, 0.0 = unrelated):")
print(f"{'='*65}")

labels = ["S0 (fee question)", "S1 (cost question)", "S2 (library hours)"]
for i in range(3):
    for j in range(i + 1, 3):
        sim_score = similarity_matrix[i][j]
        # Visually distinguish high vs. low similarity
        indicator = "✓ HIGH — similar meaning" if sim_score > 0.65 else "✗ LOW  — different topic"
        print(f"\n  {labels[i]}")
        print(f"  vs. {labels[j]}")
        print(f"  Similarity: {sim_score:.4f}  {indicator}")

# ── Key insight summary ───────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("KEY INSIGHT:")
print("  S0 and S1 use completely different words ('fee' vs 'cost')")
print("  but their vectors are CLOSE → embedding captures meaning, not words.")
print("  This is why semantic search works: a user asking 'how much' will")
print("  retrieve chunks containing 'tuition fee' — even with no word overlap.")
print()

# ── Expected output (for reference before running) ───────────────────────────
# Expected results (approximate — may vary slightly by model version):
#   S0 vs S1 similarity: ~0.88   ← HIGH: both ask about money/cost
#   S0 vs S2 similarity: ~0.28   ← LOW: fee question vs. library hours
#   S1 vs S2 similarity: ~0.31   ← LOW: cost question vs. library hours
#
# If your numbers are in this ballpark, the embedding model is working correctly.
print("Reference expected values: S0↔S1 ≈ 0.85-0.92 | S0↔S2 ≈ 0.20-0.40 | S1↔S2 ≈ 0.20-0.40")

---
## Cell 5 — Build the Vector Index

Now we combine chunking and embedding into a single step: building the **vector index**.

**What happens inside `VectorStoreIndex.from_documents()`:**
1. Each document is split into chunks using our `SentenceSplitter`
2. Each chunk is embedded (converted to a 384-dimensional vector)
3. Both the chunk text and its vector are stored together

**What gets saved to disk:**
```
data/index/
├── docstore.json      ← original text of every chunk (for displaying answers)
├── vector_store.json  ← embedding vectors (for similarity search)
└── index_store.json   ← structural metadata linking them together
```

**Why persist?** Embedding costs time (and potentially money). Saving the index means you pay this cost once, not on every query. `chatbot.py` loads these files in ~1 second at startup.

In [ ]:
from indexer import load_raw_documents, configure_embedding_model
from llama_index.core import VectorStoreIndex, Settings
from llama_index.core.node_parser import SentenceSplitter

INDEX_DIR = Path("data") / "index"
CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", "512"))
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", "50"))

# ── Configure LlamaIndex global settings ──────────────────────────────────────
print("Configuring LlamaIndex settings...")
embed_model = configure_embedding_model()
text_splitter = SentenceSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

# We set LLM to None in the indexer because we don't need text generation here.
# If we didn't do this, LlamaIndex would try to load a default LLM (OpenAI)
# and fail because no OPENAI_API_KEY is set.
Settings.embed_model = embed_model
Settings.node_parser = text_splitter
Settings.llm = None

# ── Load the raw documents from disk ─────────────────────────────────────────
print("Loading documents from data/raw/...")
documents = load_raw_documents()
print(f"Loaded {len(documents)} documents.")

# ── Build the vector index ────────────────────────────────────────────────────
print("\nBuilding vector index (chunking + embedding)...")
print("This takes 1-3 minutes. The progress bar shows chunks being embedded.")
print()

index = VectorStoreIndex.from_documents(
    documents,
    show_progress=True,  # Shows a tqdm progress bar per chunk
)

# ── Persist to disk ───────────────────────────────────────────────────────────
INDEX_DIR.mkdir(parents=True, exist_ok=True)
index.storage_context.persist(persist_dir=str(INDEX_DIR))

# ── Show what was saved ───────────────────────────────────────────────────────
total_chunks = len(index.docstore.docs)

print(f"\n{'='*55}")
print(f"Index built and saved to {INDEX_DIR}/")
print(f"{'='*55}")
print(f"Total chunks in index: {total_chunks}")
print(f"\nFiles created:")
for saved_file in sorted(INDEX_DIR.iterdir()):
    size_kb = saved_file.stat().st_size // 1024
    print(f"  {saved_file.name:<30} {size_kb:>6,} KB")

print(f"\nNext: Cell 6 shows what the retriever finds BEFORE we involve the LLM.")

---
## Cell 6 — Retrieval BEFORE Generation (The Most Important Cell)

> **This is the most important cell in the notebook.** Before we involve the LLM at all, let's see what the retriever finds.

A common misconception about RAG is that the LLM searches the knowledge base. **It doesn't.** The LLM only *generates text*. The retrieval step is pure vector math — no AI involved.

**What happens during retrieval:**
1. Your question is embedded → a 384-dimensional vector Q
2. Every chunk in the index already has its own 384-dimensional vector
3. We compute the cosine similarity between Q and every chunk vector
4. We return the top-K chunks with the highest similarity scores

Run this cell and read the raw chunks carefully. **These exact chunks are what the LLM will receive as context.** If the retriever finds irrelevant chunks, the LLM will give a poor answer — not because it's bad, but because it was given bad context. Retrieval quality is the most important factor in RAG performance.

**Try changing the question** (the `test_question` variable) and re-running to see how different questions retrieve different chunks.

In [ ]:
from llama_index.core import StorageContext, load_index_from_storage, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

INDEX_DIR = Path("data") / "index"

# ── Load the persisted index from disk ───────────────────────────────────────
# This is fast (~1 second) — no re-embedding needed.
print("Loading index from disk...")
embed_model = HuggingFaceEmbedding(model_name="all-MiniLM-L6-v2")
Settings.embed_model = embed_model
Settings.llm = None  # retrieval only — no LLM needed

storage_context = StorageContext.from_defaults(persist_dir=str(INDEX_DIR))
loaded_index = load_index_from_storage(storage_context)
print(f"Index loaded. Contains {len(loaded_index.docstore.docs)} chunks.\n")

# ── Create a retriever (NOT a full query engine) ──────────────────────────────
# A retriever does ONLY the search step — no LLM call, no answer generation.
# This lets us inspect exactly what context the LLM would receive.
retriever = loaded_index.as_retriever(
    similarity_top_k=3,  # return top 3 most similar chunks
)

# ── Ask the retriever a question ──────────────────────────────────────────────
# CHANGE THIS to see how different questions retrieve different chunks!
test_question = "What faculties and programs does Ala-Too University offer?"

print(f"Question: '{test_question}'")
print(f"Retrieving top-3 chunks...\n")

# retriever.retrieve() embeds the question and finds similar chunks.
# This is PURE vector math — no LLM involved at this step.
retrieved_nodes = retriever.retrieve(test_question)

# ── Display the retrieved chunks with their similarity scores ────────────────
print(f"{'='*65}")
print(f"RETRIEVED CHUNKS (these are sent verbatim to the LLM as context)")
print(f"{'='*65}")

for rank, node_with_score in enumerate(retrieved_nodes, start=1):
    similarity_score = node_with_score.score
    chunk_text = node_with_score.node.get_content()
    chunk_length = len(chunk_text)

    print(f"\n── CHUNK #{rank} (similarity score: {similarity_score:.4f}) ──")
    print(f"   Length: {chunk_length} characters")

    # Extract source URL from the first line (added by scraper.py)
    if chunk_text.startswith("Source URL:"):
        first_line = chunk_text.split("\n")[0]
        source_url = first_line.replace("Source URL:", "").strip()
        print(f"   Source: {source_url}")

    print(f"   Content (first 400 chars):")
    # Show the first 400 characters of the chunk, indented for readability
    chunk_preview = chunk_text[:400].replace("\n", "\n   ")
    print(f"   {chunk_preview}")
    if chunk_length > 400:
        print(f"   ... [{chunk_length - 400} more characters not shown]")

print(f"\n{'='*65}")
print("WHAT HAPPENS NEXT:")
print("  The LLM receives: [system prompt] + [these 3 chunks] + [your question]")
print("  It reads the chunks and generates an answer based ONLY on this context.")
print("  If the relevant information is not in these chunks, the LLM may say")
print("  'I don't know' — which is the correct behavior for a grounded RAG system.")

---
## Cell 7 — Full RAG Pipeline: End-to-End Q&A

Now we put everything together: **retrieve chunks** → **feed to LLM** → **get a grounded answer**.

**What the LLM actually receives** (simplified):
```
System: You are a helpful assistant for Ala-Too University.
        Answer only based on the provided context.
        If the context doesn't contain the answer, say so.

Context:
  [Chunk 1 text — 512 characters from the university website]
  [Chunk 2 text — 512 characters from the university website]
  [Chunk 3 text — 512 characters from the university website]

User: What programs does Ala-Too offer?
```

The LLM reads this and generates a response. It doesn't know anything about Ala-Too beyond what's in those 3 chunks — but that's exactly what we want! Answers are grounded in real, verifiable sources.

**Expected wait time:** 10-30 seconds per question (HuggingFace free tier). The first question may take 30-60 seconds if the model is "cold" (not loaded on any server).

In [ ]:
from chatbot import create_chat_engine, ask

# ── Verify HF_TOKEN is set before making API calls ───────────────────────────
hf_token = os.getenv("HF_TOKEN")
if not hf_token or hf_token == "hf_your_token_here":
    print("ERROR: HF_TOKEN is not set in your .env file!")
    print("Please follow the instructions in .env.example to get a free token.")
    print("Then restart the notebook kernel (Kernel → Restart) and re-run cells.")
else:
    print(f"HF_TOKEN found. Using model: {os.getenv('LLM_MODEL_ID', 'mistralai/Mistral-7B-Instruct-v0.3')}")
    print()

    # ── Create the full RAG chat engine ──────────────────────────────────────
    # This loads: embedding model + vector index + configures HF Inference API LLM
    print("Creating chat engine (loading index + configuring LLM)...")
    chat_engine = create_chat_engine()
    print()

    # ── Ask 3 real questions about Ala-Too University ─────────────────────────
    real_questions = [
        "What faculties and academic programs does Ala-Too International University offer?",
        "How can a prospective student apply for admission to Ala-Too University?",
        "Tell me about the history and mission of Ala-Too International University.",
    ]

    for question_number, question in enumerate(real_questions, start=1):
        print(f"\n{'='*65}")
        print(f"QUESTION {question_number}: {question}")
        print(f"{'='*65}")
        print("Retrieving chunks + calling Mistral-7B API...")
        print("(This may take 10-30 seconds for the API call)\n")

        # ask() calls: retrieve chunks → format prompt → call HF API → parse response
        answer_text, source_urls = ask(chat_engine, question)

        print(f"ANSWER:")
        print(answer_text)

        print(f"\nSOURCES ({len(source_urls)} page{'s' if len(source_urls) != 1 else ''}):")
        if source_urls:
            for source_url in source_urls:
                print(f"  → {source_url}")
        else:
            print("  (No source URLs extracted — check scraper.py output format)")

    # ── Demonstrate multi-turn conversation ───────────────────────────────────
    print(f"\n{'='*65}")
    print("BONUS: Multi-turn conversation (condense_plus_context mode)")
    print(f"{'='*65}")
    print("The chat engine remembers previous turns. Follow-up questions")
    print("are condensed into standalone questions using the history.\n")

    followup_question = "What language are the programs taught in?"
    print(f"Follow-up question: '{followup_question}'")
    print("(Internally, the engine rewrites this using the prior conversation context)\n")
    followup_answer, followup_sources = ask(chat_engine, followup_question)
    print(f"Answer: {followup_answer}")

---
## Cell 8 — Launch the Streamlit App

The full pipeline is working. Now let's launch the browser-based chat interface.

**How to run the app:**

Open a **new terminal window** (keep this notebook open), navigate to the project directory, and run:

```bash
streamlit run app.py
```

Streamlit will print a local URL (usually `http://localhost:8501`) — open it in your browser.

**Why a separate terminal?**
Streamlit is a long-running web server. Running it from the notebook cell would block the notebook kernel until you stop it. Running it in a separate terminal lets you keep using the notebook while the app runs.

The cell below launches the app programmatically — this is useful for demos but will block the current notebook kernel. **Use the terminal command above for normal use.**

In [ ]:
import subprocess
import sys
import time

# ── Check that app.py exists before trying to launch it ──────────────────────
app_path = Path("app.py")
if not app_path.exists():
    print("ERROR: app.py not found in the current directory.")
    print(f"Current directory: {Path.cwd()}")
    print("Make sure you're running this notebook from the ala_too_chatbot/ folder.")
else:
    print("="*55)
    print("  Ala-Too University RAG Chatbot — Streamlit App")
    print("="*55)
    print()
    print("OPTION A (recommended): Run in a separate terminal:")
    print()
    print("    streamlit run app.py")
    print()
    print("Then open: http://localhost:8501")
    print()
    print("-"*55)
    print("OPTION B: Launch from this cell (blocks this notebook):")
    print()

    # Check if the user wants to launch now
    # (Change LAUNCH_NOW to True if you want to start from this cell)
    LAUNCH_NOW = False  # ← Change to True to launch Streamlit from this cell

    if LAUNCH_NOW:
        print("Launching Streamlit...")
        print("To stop: interrupt the kernel (■ button) or Ctrl+C in terminal.")
        print()
        print("→ Opening: http://localhost:8501")
        print()

        # subprocess.run() launches streamlit as a child process.
        # This WILL block the notebook cell until streamlit is stopped.
        # Use Kernel → Interrupt (or the ■ button) to stop it.
        subprocess.run(
            [sys.executable, "-m", "streamlit", "run", "app.py",
             "--server.headless", "true"],
            check=False,
        )
    else:
        print("LAUNCH_NOW = False. Change to True in the cell above to launch here.")
        print()
        print("Or run in your terminal:")
        print("    streamlit run app.py")
        print()
        print("="*55)
        print("Congratulations! You've built a complete RAG chatbot.")
        print()
        print("What you accomplished in this notebook:")
        print("  1. Scraped 20 pages from ala-too.edu.kg")
        print("  2. Split text into overlapping chunks (512 chars, 50 overlap)")
        print("  3. Embedded each chunk with all-MiniLM-L6-v2 (384 dimensions)")
        print("  4. Built and persisted a vector index")
        print("  5. Retrieved top-3 chunks using cosine similarity")
        print("  6. Sent chunks + question to Mistral-7B via HF Inference API")
        print("  7. Got grounded, cited answers")
        print("="*55)